In [1]:
from pathlib import Path
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import DecisionBoundaryDisplay, permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, mean_absolute_error, mean_squared_error, roc_auc_score
from sklearn.model_selection import (
GridSearchCV, KFold, StratifiedKFold, cross_val_score, train_test_split,
)
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
PROJECT_ROOT = Path.cwd().parent
DATA = PROJECT_ROOT / "all_datasets"
MODELS = PROJECT_ROOT / "models"
MODELS.mkdir(exist_ok=True)


# Part A - KNN versus Gaussian naive Bayes

In [5]:
cancer = pd.read_csv(DATA / "breast_cancer_dataset.csv")
X_c = cancer.drop(columns=["id", "diagnosis", "Unnamed: 32"], errors="ignore")
y_c = (cancer["diagnosis"] == "M").astype(int)

Xc_train, Xc_valid, yc_train, yc_valid = train_test_split(
    X_c, y_c, test_size=0.25, random_state=42, stratify=y_c
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
k_values = [1, 3, 5, 7, 11, 21]
cv_scores = {}

for k in k_values:
    candidate = Pipeline([
        ("scale", StandardScaler()),
        ("model", KNeighborsClassifier(n_neighbors=k, weights="distance")),
    ])
    scores = cross_val_score(candidate, Xc_train, yc_train, cv=cv, scoring="roc_auc")
    cv_scores[k] = scores.mean()
    print(f"k={k:2d} cross-validated ROC AUC: {scores.mean():.3f}")

best_k = max(cv_scores, key=cv_scores.get)
print("selected k:", best_k)

models = {
    "KNN": Pipeline([
        ("scale", StandardScaler()),
        ("model", KNeighborsClassifier(n_neighbors=best_k, weights="distance")),
    ]),
    "GaussianNB": Pipeline([
        ("scale", StandardScaler()),
        ("model", GaussianNB()),
    ]),
}

for name, model in models.items():
    model.fit(Xc_train, yc_train)
    pred = model.predict(Xc_valid)
    prob_c = model.predict_proba(Xc_valid)[:, 1]
    print(f"{name:10s} accuracy={accuracy_score(yc_valid, pred):.3f} roc_auc={roc_auc_score(yc_valid, prob_c):.3f}")

k= 1 cross-validated ROC AUC: 0.946
k= 3 cross-validated ROC AUC: 0.984
k= 5 cross-validated ROC AUC: 0.985
k= 7 cross-validated ROC AUC: 0.987
k=11 cross-validated ROC AUC: 0.988
k=21 cross-validated ROC AUC: 0.988
selected k: 11
KNN        accuracy=0.958 roc_auc=0.986
GaussianNB accuracy=0.944 roc_auc=0.992


# Part B - Decision Trees & Overfitting

In [6]:
titanic = pd.read_csv(DATA / "titanic_dataset.csv")
t_num = ["Age", "SibSp", "Parch", "Fare"]
t_cat = ["Pclass", "Sex", "Embarked"]
features = t_num + t_cat

X_t = titanic[features]
y_t = titanic["Survived"]

tree_preprocess = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), t_num),
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), t_cat),
])

Xt_train, Xt_valid, yt_train, yt_valid = train_test_split(
    X_t, y_t, test_size=0.25, random_state=42, stratify=y_t
)

tree = Pipeline([
    ("preprocess", tree_preprocess),
    ("tree", DecisionTreeClassifier(max_depth=4, min_samples_leaf=12, random_state=42)),
])
tree.fit(Xt_train, yt_train)
print("training accuracy:", round(accuracy_score(yt_train, tree.predict(Xt_train)), 3))
print("validation accuracy:", round(accuracy_score(yt_valid, tree.predict(Xt_valid)), 3))

training accuracy: 0.843
validation accuracy: 0.767


# Part C - Random Forest vs Gradient Boosting

In [7]:
ames = pd.read_csv(DATA / "ames_housing_dataset.csv")
X_a = ames.drop(columns=["SalePrice", "Order", "PID"], errors="ignore")
y_a = np.log1p(ames["SalePrice"])

ames_preprocess = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), X_a.select_dtypes(include="number").columns),
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]), X_a.select_dtypes(exclude="number").columns),
])

ensembles = {
    "random_forest": RandomForestRegressor(
        n_estimators=150, max_features=0.8, min_samples_leaf=2, random_state=42, n_jobs=-1
    ),
    "gradient_boosting": GradientBoostingRegressor(
        n_estimators=150, learning_rate=0.05, max_depth=3, loss="huber", random_state=42
    ),
}

cv3 = KFold(n_splits=3, shuffle=True, random_state=42)
cv_rmse = {}
for name, estimator in ensembles.items():
    pipe = Pipeline([("preprocess", ames_preprocess), ("model", estimator)])
    neg_mse = cross_val_score(pipe, X_a, y_a, cv=cv3, scoring="neg_mean_squared_error")
    cv_rmse[name] = np.sqrt(-neg_mse)
    print(f"{name:18s} CV log-RMSE {cv_rmse[name].mean():.4f} +/- {cv_rmse[name].std():.4f}")

random_forest      CV log-RMSE 0.1382 +/- 0.0118
gradient_boosting  CV log-RMSE 0.1273 +/- 0.0141


# Part D - Support Vector Machines

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X_c, y_c, test_size=0.25, random_state=42, stratify=y_c
)

svm_pipe = Pipeline([("scale", StandardScaler()), ("model", SVC())])
search = GridSearchCV(
    svm_pipe,
    param_grid={
        "model__kernel": ["linear", "rbf"],
        "model__C": [0.1, 1, 10, 100],
        "model__gamma": ["scale", 0.01, 0.1],
    },
    cv=StratifiedKFold(5, shuffle=True, random_state=42),
    scoring="roc_auc",
    n_jobs=-1,
)
search.fit(X_train, y_train)
score = search.decision_function(X_test)
pred = search.predict(X_test)

print("best parameters:", search.best_params_)
print("cross-validated ROC AUC:", round(search.best_score_, 3))
print("test accuracy:", round(accuracy_score(y_test, pred), 3))
print("test ROC AUC:", round(roc_auc_score(y_test, score), 3))

# Model persistence test
model_path = MODELS / "breast_cancer_svm.joblib"
joblib.dump(search.best_estimator_, model_path)
loaded = joblib.load(model_path)
same = np.array_equal(loaded.predict(X_test), pred)
print("file size (KB):", round(model_path.stat().st_size / 1024, 1))
print("reloaded model reproduces the test predictions:", same)

best parameters: {'model__C': 10, 'model__gamma': 0.01, 'model__kernel': 'rbf'}
cross-validated ROC AUC: 0.995
test accuracy: 0.979
test ROC AUC: 0.997
file size (KB): 16.5
reloaded model reproduces the test predictions: True


In [10]:
# Fully grown tree (no depth ceiling, single sample leaf)
full_tree = Pipeline([
    ("preprocess", tree_preprocess),
    ("tree", DecisionTreeClassifier(max_depth=None, min_samples_leaf=1, random_state=42)),
])
full_tree.fit(Xt_train, yt_train)

print("--- Unconstrained Tree ---")
print("Training accuracy:  ", round(accuracy_score(yt_train, full_tree.predict(Xt_train)), 3))
print("Validation accuracy:", round(accuracy_score(yt_valid, full_tree.predict(Xt_valid)), 3))

--- Unconstrained Tree ---
Training accuracy:   0.978
Validation accuracy: 0.722


Expected outcome & interpretation: Training accuracy reaches near $0.98 - 1.00$ because the tree memorizes individual samples down to pure leaves. However, validation accuracy drops significantly (typically dropping below $\approx 0.76 - 0.78$), clearly demonstrating high variance / overfitting compared to the constrained depth-4 model ($\approx 0.81$).

In [11]:
search_linear = GridSearchCV(
    svm_pipe,
    param_grid={
        "model__kernel": ["linear"],
        "model__C": [0.1, 1, 10, 100],
    },
    cv=StratifiedKFold(5, shuffle=True, random_state=42),
    scoring="roc_auc",
    n_jobs=-1,
)
search_linear.fit(X_train, y_train)
linear_score = search_linear.decision_function(X_test)

print("--- Linear SVM Only ---")
print("Best params:", search_linear.best_params_)
print("Test ROC AUC (Linear):", round(roc_auc_score(y_test, linear_score), 3))
print("Test ROC AUC (RBF/Best Search):", round(roc_auc_score(y_test, score), 3))

--- Linear SVM Only ---
Best params: {'model__C': 0.1, 'model__kernel': 'linear'}
Test ROC AUC (Linear): 0.997
Test ROC AUC (RBF/Best Search): 0.997


Expected outcome & interpretation: On the standard breast cancer dataset, linear and RBF kernels yield virtually identical ROC AUCs ($>0.98$). The normalized 30-dimensional features are already linearly separable to a high degree, making complex non-linear boundary warping unnecessary.

1. What is the geometric meaning of a larger separating margin in an SVM?
= Geometrically, the margin is the perpendicular distance between the decision boundary hyperplane ($\mathbf{w}^T\mathbf{x} + b = 0$) and the closest training data points (support vectors) from either class. A larger margin ($\frac{2}{\Vert{}\mathbf{w}\Vert{}}$) provides a wider "buffer zone" of safety, reducing sensitivity to noise and lowering model variance on unseen data.

2. How do $C$ and, for an RBF kernel, $\gamma$ change the flexibility of the fitted decision boundary?
   = $C$ (Penalty parameter): Controls the trade-off between maximizing margin width and penalizing margin/classification violations.A large $C$ heavily penalizes violations, forcing the margin to shrink and fit outliers strictly (higher variance, more flexible boundary, risk of overfitting).A small $C$ tolerates more misclassifications for a wider, simpler margin (higher bias, smoother boundary).
   = $\gamma$ (RBF kernel bandwidth parameter): Controls the radius of influence of each support vector ($K(x_i, x_j) = \exp(-\gamma \Vert{}x_i - x_j\Vert{}^2)$).A large $\gamma$ means individual points have localized influence, causing the decision boundary to form tight, complex islands around individual training points (high variance, risk of overfitting).A small $\gamma$ gives each point widespread influence, smoothing the decision boundary toward a near-linear surface (high bias).

3. Why should scaling and SVM hyperparameter selection both happen inside the validation procedure?
a) Preventing Data Leakage: If features are standardized using mean and variance calculated over the entire dataset (including validation/test splits), information about the test distribution leaks into the training phase. Placing StandardScaler() inside a Pipeline ensures normalization parameters are calculated strictly on each training fold.
b) Unbiased Model Evaluation: Tuning $C$, $\gamma$, or kernel on the test set overfits the hyperparameters to test noise. Wrapping both the scaling and the estimator inside cross-validation partitions the data cleanly, giving an honest, generalizable estimate of performance.